# Memory Estimates Playground — coding workshop (advanced)

**The Colab version of the lesson exercise "Which models fit on a 16 GB GPU?"**
Expected time: ~8–10 minutes, individual, no GPU needed.

The lesson asks you to estimate the weight memory for 7B, 32B, and 70B-parameter models in
FP16, INT8, and INT4, using `memory = parameters * bits / 8`, and to check whether each
result would fit into 16 GB of GPU memory. Here you do that same calculation in code
instead of on paper, and check it against a model size the lesson already worked out for
you before moving on to the bigger ones.


## How this workshop works

This is the coding version of the notebook, **advanced level**. In some code cells, lines
are replaced by `# TODO` comments and `___`. Each exercise above such a cell says what the
code must do; the comments in the cell give the details. This version takes longer than
the expected time given above.

1. Replace every `___` with your code, then run the cell. Open the hints if you need them.
2. Run the check cell right below it. It prints `✓ Exercise N passed`, or an error
   message that says what to look at.
3. If you are stuck, the solution is below the check cell. Open it after you have tried.

A cell that still contains `___` stops with `NameError: name '___' is not defined`.
Later cells use your code, so finish each exercise before moving on.

Exercises marked **(advanced)** are only in this version. In them you implement an
equation from the lesson or a paper, or a technique that makes the pipeline cheaper to
run, and check it against the library or the baseline.

## Step 1 — the formula, checked against a number you already have

The lesson screen "Memory for the weights of our model" already gave you the answer for
Qwen2.5-1.5B: about 6 GB at FP32, 3 GB at FP16/BF16, 1.5 GB at INT8, 0.75 GB at INT4. Run
the same formula here for that same model size, as a sanity check that the formula and
the lesson agree before you use it on a new number.


### ✏️ Exercise 1 — Weight memory from parameter count and bits

Write `weight_memory_gb`: the memory for the weights alone, in GB (1 GB = 10⁹
bytes, as on the lesson's memory screen), for a model with `params_billion` billion parameters stored at `bits` bits
each.

<details><summary><b>Hints</b> (click to open)</summary>

- One billion is `1e9`.
- There are 8 bits in a byte.
- 1 GB here is 10⁹ bytes, so our 1.5B model in 16 bits comes out at exactly 3 GB.

</details>

In [ ]:
def weight_memory_gb(params_billion, bits):
    params = ___  # TODO: number of parameters
    bytes_total = ___  # TODO: total bytes
    return ___  # TODO: in GB (10**9 bytes)


for bits, label in [(32, "FP32"), (16, "FP16/BF16"), (8, "INT8"), (4, "INT4")]:
    gb = weight_memory_gb(1.5, bits)
    print(f"1.5B model, {label}: {gb:.2f} GB  (lesson: ~{ {32:6, 16:3, 8:1.5, 4:0.75}[bits] } GB)")


In [ ]:
# Check for Exercise 1. Run this cell as is.
assert abs(weight_memory_gb(1.5, 16) - 3.0) < 1e-9, weight_memory_gb(1.5, 16)
assert abs(weight_memory_gb(7, 4) - 3.5) < 1e-9, weight_memory_gb(7, 4)
assert abs(weight_memory_gb(1.5, 32) / weight_memory_gb(1.5, 8) - 4) < 1e-9
print("✓ Exercise 1 passed")

<details><summary><b>Solution to Exercise 1</b> (open it after you have tried)</summary>

```python
def weight_memory_gb(params_billion, bits):
    params = params_billion * 1e9
    bytes_total = params * bits / 8
    return bytes_total / 1e9  # 1 GB = 10**9 bytes


for bits, label in [(32, "FP32"), (16, "FP16/BF16"), (8, "INT8"), (4, "INT4")]:
    gb = weight_memory_gb(1.5, bits)
    print(f"1.5B model, {label}: {gb:.2f} GB  (lesson: ~{ {32:6, 16:3, 8:1.5, 4:0.75}[bits] } GB)")
```

</details>

## Step 2 — now the actual activity: 7B, 32B, and 70B on a 16 GB GPU

A Colab T4 has 16 GB of GPU memory. `GPU_MEMORY_GB` below is deliberately just that one
number, ignoring activations, the KV cache, and framework overhead, exactly as the lesson
says to for this first pass. 7B repeats the earlier point about headroom; 32B and 70B are
where the question stops being "does quantization help" and starts being "does the model
fit at all."


### ✏️ Exercise 2 — Does it fit, and at what precision?

Two parts:

1. Decide whether each configuration fits in `GPU_MEMORY_GB`.
2. Write `smallest_fitting_bits(model_size)`: go through the precisions in `options`, from
   most to fewest bits, and return the first one whose weights fit. Return `None` if none
   fits.

<details><summary><b>Hints</b> (click to open)</summary>

- Use `weight_memory_gb(model_size, bits)` from Step 1.
- Return inside the loop as soon as one fits; the loop goes from most to fewest bits.

</details>

In [ ]:
GPU_MEMORY_GB = 16  # a Colab T4

for model_size in [7, 32, 70]:
    for bits, label in [(16, "FP16"), (8, "INT8"), (4, "INT4")]:
        gb = weight_memory_gb(model_size, bits)
        fits = ___  # TODO: "fits" or "does NOT fit"
        print(f"{model_size}B model, {label}: {gb:.2f} GB --- {fits} in {GPU_MEMORY_GB} GB")
    print()


def smallest_fitting_bits(model_size, gpu_gb=GPU_MEMORY_GB, options=(16, 8, 4, 3, 2)):
    # TODO: the first bit width in options whose weights fit in gpu_gb, or None
    ___


for model_size in [7, 32, 70]:
    print(f"{model_size}B: highest precision that fits = {smallest_fitting_bits(model_size)} bits")


In [ ]:
# Check for Exercise 2. Run this cell as is.
assert [smallest_fitting_bits(s) for s in (7, 32, 70)] == [16, 3, None], \
    [smallest_fitting_bits(s) for s in (7, 32, 70)]
assert smallest_fitting_bits(70, gpu_gb=80) == 8
print("✓ Exercise 2 passed")

<details><summary><b>Solution to Exercise 2</b> (open it after you have tried)</summary>

```python
GPU_MEMORY_GB = 16  # a Colab T4

for model_size in [7, 32, 70]:
    for bits, label in [(16, "FP16"), (8, "INT8"), (4, "INT4")]:
        gb = weight_memory_gb(model_size, bits)
        fits = "fits" if gb < GPU_MEMORY_GB else "does NOT fit"
        print(f"{model_size}B model, {label}: {gb:.2f} GB --- {fits} in {GPU_MEMORY_GB} GB")
    print()


def smallest_fitting_bits(model_size, gpu_gb=GPU_MEMORY_GB, options=(16, 8, 4, 3, 2)):
    for bits in options:
        if weight_memory_gb(model_size, bits) < gpu_gb:
            return bits
    return None


for model_size in [7, 32, 70]:
    print(f"{model_size}B: highest precision that fits = {smallest_fitting_bits(model_size)} bits")
```

</details>

### ✏️ Exercise 3 — The KV cache formula (advanced)

Weights are not the only thing in GPU memory. While a model generates, it keeps a
key vector and a value vector for every token seen so far, in every layer: the **KV cache**.
Its size is

KV cache bytes = 2 · layers · kv_heads · head_dim · seq_len · batch · bits / 8

The 2 counts keys and values. `kv_heads` is the number of key/value heads; recent models
use fewer of them than query heads, to keep the cache small. `seq_len` is the number of
tokens in each sequence and `batch` the number of sequences.

Complete `kv_cache_gb`, in GB of 1024³ bytes like `weight_memory_gb`.

<details><summary><b>Hints</b> (click to open)</summary>

- Multiply all the factors, then divide by 8 (bits to bytes) and by 1024 ** 3 (bytes to GB).
- Qwen2.5-1.5B: 2 · 28 · 2 · 128 · 2 bytes = 28,672 bytes for each token.

</details>

In [ ]:
def kv_cache_gb(layers, kv_heads, head_dim, seq_len, batch, bits=16):
    # TODO: the KV cache formula, in GB (1024**3 bytes)
    ___

# From the models' config.json files on the Hugging Face Hub.
MODELS = {
    "Qwen2.5-1.5B": {"params_billion": 1.54, "layers": 28, "kv_heads": 2, "head_dim": 128},
    "Qwen2.5-7B": {"params_billion": 7.61, "layers": 28, "kv_heads": 4, "head_dim": 128},
}

for name, m in MODELS.items():
    per_token = kv_cache_gb(m["layers"], m["kv_heads"], m["head_dim"], seq_len=1, batch=1, bits=16)
    print(f"{name}: KV cache {per_token * 1024**2:.0f} KB per token, "
          f"{kv_cache_gb(m['layers'], m['kv_heads'], m['head_dim'], seq_len=32768, batch=1, bits=16):.2f} GB "
          "for one 32,768-token sequence")

In [ ]:
# Check for Exercise 3. Run this cell as is.
assert abs(kv_cache_gb(28, 2, 128, 1, 1) * 1024 ** 3 - 28672) < 1e-6, "Qwen2.5-1.5B: 28,672 bytes per token"
assert abs(kv_cache_gb(28, 4, 128, 32768, 1) - 1.75) < 1e-9
assert kv_cache_gb(28, 4, 128, 1000, 8) == 8 * kv_cache_gb(28, 4, 128, 1000, 1)
print("✓ Exercise 3 passed")

<details><summary><b>Solution to Exercise 3</b> (open it after you have tried)</summary>

```python
def kv_cache_gb(layers, kv_heads, head_dim, seq_len, batch, bits=16):
    return 2 * layers * kv_heads * head_dim * seq_len * batch * bits / 8 / (1024 ** 3)

# From the models' config.json files on the Hugging Face Hub.
MODELS = {
    "Qwen2.5-1.5B": {"params_billion": 1.54, "layers": 28, "kv_heads": 2, "head_dim": 128},
    "Qwen2.5-7B": {"params_billion": 7.61, "layers": 28, "kv_heads": 4, "head_dim": 128},
}

for name, m in MODELS.items():
    per_token = kv_cache_gb(m["layers"], m["kv_heads"], m["head_dim"], seq_len=1, batch=1, bits=16)
    print(f"{name}: KV cache {per_token * 1024**2:.0f} KB per token, "
          f"{kv_cache_gb(m['layers'], m['kv_heads'], m['head_dim'], seq_len=32768, batch=1, bits=16):.2f} GB "
          "for one 32,768-token sequence")
```

</details>

### ✏️ Exercise 4 — The longest context that fits (advanced)

With the weights loaded, whatever GPU memory is left can hold the KV cache. Solve
the memory equation for the number of tokens:

weights + KV cache per token · tokens ≤ GPU memory

Complete `max_context_tokens` (one sequence: the longest context that fits) and
`max_batch` (how many sequences of `seq_len` tokens fit at once). Both return 0 if the
weights alone do not fit. The KV cache stays at 16 bits here, as is common, even when the
weights are 4-bit. As in Step 2, this ignores activations and framework overhead, so the
real limits are lower.

**Reflection:** Qwen2.5 models are trained for contexts of up to 32,768 tokens. For which
configurations is the GPU the limit, and for which is it the model?

<details><summary><b>Hints</b> (click to open)</summary>

- Free memory divided by the memory per token is the number of tokens; round down with `math.floor`.
- If `free_gb` is negative, `max(0, ...)` returns 0.
- A batch of sequences needs `batch * seq_len` tokens in the cache.

</details>

In [ ]:
import math


def max_context_tokens(m, weight_bits, gpu_gb):
    free_gb = gpu_gb - weight_memory_gb(m["params_billion"], weight_bits)
    per_token_gb = kv_cache_gb(m["layers"], m["kv_heads"], m["head_dim"], seq_len=1, batch=1)
    # TODO: the number of whole tokens that fit in free_gb (0 if nothing fits)
    ___


def max_batch(m, weight_bits, gpu_gb, seq_len):
    # TODO: the number of whole sequences of seq_len tokens that fit (0 if nothing fits)
    ___

print(f"{'model':14s} {'weights':>8s} {'longest context':>16s} {'batch at 4,096 tokens':>22s}")
for name, m in MODELS.items():
    for bits, label in [(16, "FP16"), (4, "INT4")]:
        n = max_context_tokens(m, bits, GPU_MEMORY_GB)
        b = max_batch(m, bits, GPU_MEMORY_GB, seq_len=4096)
        print(f"{name:14s} {label:>8s} {n:16,d} {b:22,d}")

In [ ]:
# Check for Exercise 4. Run this cell as is.
_m = MODELS["Qwen2.5-7B"]
_n = max_context_tokens(_m, 16, 16)
_per = kv_cache_gb(28, 4, 128, 1, 1)
assert weight_memory_gb(7.61, 16) + _n * _per <= 16 < weight_memory_gb(7.61, 16) + (_n + 1) * _per
assert max_context_tokens(_m, 16, 8) == 0, "the 7B weights alone do not fit in 8 GB"
assert max_batch(_m, 4, 16, 4096) == max_context_tokens(_m, 4, 16) // 4096
print("✓ Exercise 4 passed")

<details><summary><b>Solution to Exercise 4</b> (open it after you have tried)</summary>

```python
import math


def max_context_tokens(m, weight_bits, gpu_gb):
    free_gb = gpu_gb - weight_memory_gb(m["params_billion"], weight_bits)
    per_token_gb = kv_cache_gb(m["layers"], m["kv_heads"], m["head_dim"], seq_len=1, batch=1)
    return max(0, math.floor(free_gb / per_token_gb))


def max_batch(m, weight_bits, gpu_gb, seq_len):
    return max_context_tokens(m, weight_bits, gpu_gb) // seq_len

print(f"{'model':14s} {'weights':>8s} {'longest context':>16s} {'batch at 4,096 tokens':>22s}")
for name, m in MODELS.items():
    for bits, label in [(16, "FP16"), (4, "INT4")]:
        n = max_context_tokens(m, bits, GPU_MEMORY_GB)
        b = max_batch(m, bits, GPU_MEMORY_GB, seq_len=4096)
        print(f"{name:14s} {label:>8s} {n:16,d} {b:22,d}")
```

</details>

**Reflection:** Look at your printed results for 32B and 70B. At 32B, which precisions fit
and which don't? At 70B? What does this tell you about when quantization changes from
"helps efficiency" to "the only way to fit the model at all" --- and is there a point where
quantization alone isn't enough either?


## Code literacy check

Before you move on, add `#` comments to the two code cells above explaining, in your own
words, what each line does and why it's there, in language a classmate who hasn't seen
the code could follow.

Do this part on your own, after class:

1. Paste your commented code into an LLM chat (ChatGPT, Claude, Gemini — whichever you
   have access to).
2. Ask it something like: *"Here is code I commented myself, explaining what I think it
   does. Based only on my comments, how well do you think I understood this code? What
   did I get right, what did I miss, and what's still fuzzy?"*
3. Read what it says. Where it disagrees with your comment, go check the code again —
   who's right?
